# 02 — Feature Engineering

Build causal 1H microstructure features and correctly delayed 4H features.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import pandas as pd
from yenibot.features import build_feature_matrix

primary = pd.read_parquet(f'{DATA_DIR}/raw/btc_1h.parquet')
htf = pd.read_parquet(f'{DATA_DIR}/raw/btc_4h.parquet')
intrabar = None
futures_metrics = None
funding_rates = None

binance = cfg.get('binance', {})
intrabar_intervals = binance.get('intrabar_intervals', [])
if intrabar_intervals:
    intrabar_path = f"{DATA_DIR}/raw/btc_{intrabar_intervals[0]}.parquet"
    if os.path.exists(intrabar_path):
        intrabar = pd.read_parquet(intrabar_path)
        print('Loaded intrabar data:', intrabar_path, len(intrabar))
    else:
        print('Intrabar data missing; 15m profiles will run without ih15 features until 01 is rerun:', intrabar_path)

metrics_cfg = binance.get('futures_metrics', {}) or {}
if metrics_cfg.get('enabled', False):
    metrics_path = f"{DATA_DIR}/raw/{metrics_cfg.get('filename', 'btc_futures_metrics.parquet')}"
    if os.path.exists(metrics_path):
        futures_metrics = pd.read_parquet(metrics_path)
        print('Loaded futures metrics:', metrics_path, len(futures_metrics))
    else:
        print('Futures metrics missing; futures-context profiles will run without fut metrics until 01 is rerun:', metrics_path)

funding_cfg = binance.get('funding_rates', {}) or {}
if funding_cfg.get('enabled', False):
    funding_path = f"{DATA_DIR}/raw/{funding_cfg.get('filename', 'btc_funding_rates.parquet')}"
    if os.path.exists(funding_path):
        funding_rates = pd.read_parquet(funding_path)
        print('Loaded funding rates:', funding_path, len(funding_rates))
    else:
        print('Funding rates missing; funding features will be absent until REST access is available:', funding_path)

features = build_feature_matrix(
    primary,
    htf,
    cfg,
    intrabar_frame=intrabar,
    futures_metrics_frame=futures_metrics,
    funding_frame=funding_rates,
)
out = f'{DATA_DIR}/processed/features_1h.parquet'
features.frame.to_parquet(out, index=False)
print('Saved', out)
print('Feature count:', len(features.feature_columns))
print('Date range:', features.frame['timestamp'].min(), '->', features.frame['timestamp'].max())
print('Memory MB:', features.frame.memory_usage(deep=True).sum() / 1024**2)
assert not features.frame[features.feature_columns].isna().any().any()


In [ ]:
from google.colab import runtime
runtime.unassign()